In [1]:
!pip install transformers torch Pillow

In [2]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"사용 중인 장치: {device}")  # cuda 뜨면 성공!

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

사용 중인 장치: cuda


In [3]:
def get_similarity(image_path, texts):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(
        text=texts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=1)
    result = dict(zip(texts, probs[0].tolist()))
    return result

In [4]:
texts = [
    "a casual white t-shirt",
    "a formal black suit",
    "a street style hoodie",
    "a floral summer dress"
]

result = get_similarity("/sh_wh.jpeg", texts)

for text, score in sorted(result.items(), key=lambda x: -x[1]):
    print(f"{score:.3f} | {text}")

FileNotFoundError: [Errno 2] No such file or directory: '/sh_wh.jpeg'

In [5]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 14.3 MB/s eta 0:00:00


In [13]:
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API_KEY")

def extract_fashion_keywords(query: str) -> list:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": f"""
너는 패션 전문가야.
사용자가 입력한 스타일 요청을 분석해서
CLIP 이미지 검색에 쓸 핵심 영어 키워드 5~10개를 추출해줘.
스타일, 소재, 핏, 색감, 상황을 고려해서.

입력: "{query}"

출력 형식: 영어 키워드만 콤마로 구분 (예: casual, oversized, cotton, bright color, daily)
다른 말은 하지 말고 키워드만 출력해.
"""
            }
        ]
    )
    keywords = response.choices[0].message.content.strip().split(",")
    return [k.strip() for k in keywords]

In [14]:
result = extract_fashion_keywords("오늘 좀 힙하게 입고 싶어")
print(result)

result2 = extract_fashion_keywords("결혼식 하객룩 추천해줘")
print(result2)

['street fashion', 'trendy', 'casual', 'oversized', 'bold color', 'urban', 'hip hop', 'edgy', 'youthful']
['formal', 'wedding', 'guest', 'dressy', 'elegant', 'sophisticated', 'bright color', 'luxurious', 'lace', 'satin']


In [6]:
def recommend_by_query(query: str, image_paths: list):
    print(f"🔍 입력: {query}")

    # 1. LLM으로 키워드 추출
    keywords = extract_fashion_keywords(query)
    print(f"📝 추출된 키워드: {keywords}")

    # 2. 키워드를 하나의 텍스트로 합치기
    text_query = ", ".join(keywords)

    # 3. 모든 이미지를 한번에 넣어서 비교 (softmax가 의미있어짐!)
    images = [Image.open(p).convert("RGB") for p in image_paths]
    inputs = processor(
        text=[text_query],
        images=images,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # 이미지 여러 장 vs 텍스트 1개
    probs = outputs.logits_per_text.softmax(dim=1)[0]

    scores = dict(zip(image_paths, probs.tolist()))
    ranked = sorted(scores.items(), key=lambda x: -x[1])

    print(f"\n🏆 추천 결과:")
    for i, (path, score) in enumerate(ranked):
        print(f"  {i+1}위: {path} (유사도: {score:.3f})")

    return ranked

In [ ]:
# 올려놓은 사진 경로 목록
my_clothes = [
    "/sh_wh.jpeg",
    "/jean_blue.jpeg",   # 네가 올린 파일명으로 바꿔
    "/jk_bl.jpeg",
]

recommend_by_query("오늘 좀 힙하게 입고 싶어", my_clothes)

🔍 입력: 오늘 좀 힙하게 입고 싶어
📝 추출된 키워드: ['street fashion', 'trendy', 'casual', 'oversized', 'distressed denim', 'bold color', 'edgy', 'urban', 'stylish']

🏆 추천 결과:
  1위: /jean_blue.jpeg (유사도: 0.843)
  2위: /jk_bl.jpeg (유사도: 0.115)
  3위: /sh_wh.jpeg (유사도: 0.042)


[('/jean_blue.jpeg', 0.8430987000465393),
 ('/jk_bl.jpeg', 0.11473783105611801),
 ('/sh_wh.jpeg', 0.0421634204685688)]

In [7]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 101.8 MB/s eta 0:00:00


In [9]:
import faiss
import json
import numpy as np

# 인덱스 & 경로 불러오기
index_top = faiss.read_index("/content/top_db.index")
index_bottom = faiss.read_index("/content/bottom_db.index")
index_dress = faiss.read_index("/content/dress_db.index")
index_outer = faiss.read_index("/content/outer_db.index")

with open("/content/top_paths.json") as f:
    top_paths = json.load(f)
with open("/content/bottom_paths.json") as f:
    bottom_paths = json.load(f)
with open("/content/dress_paths.json") as f:
    dress_paths = json.load(f)
with open("/content/outer_paths.json") as f:

    outer_paths = json.load(f)

print("인덱스 로드 완료!")
print(f"상의: {index_top.ntotal}벌, 하의: {index_bottom.ntotal}벌, 원피스: {index_dress.ntotal}벌, 아우터: {index_outer.ntotal}벌")

인덱스 로드 완료!
상의: 1벌, 하의: 1벌, 원피스: 1벌, 아우터: 1벌


In [10]:
print(f"상의: {index_top.ntotal}벌")
print(f"하의: {index_bottom.ntotal}벌")
print(f"원피스: {index_dress.ntotal}벌")
print(f"아우터: {index_outer.ntotal}벌")

print("\n--- 상의 경로 목록 ---")
for p in top_paths:
    print(p)

print("\n--- 하의 경로 목록 ---")
for p in bottom_paths:
    print(p)

상의: 1벌
하의: 1벌
원피스: 1벌
아우터: 1벌

--- 상의 경로 목록 ---
/content/drive/MyDrive/패션시멘틱검색/옷/캐주얼위.webp

--- 하의 경로 목록 ---
/content/drive/MyDrive/패션시멘틱검색/옷/청바지.webp


In [23]:
from transformers import CLIPModel, CLIPProcessor
import torch

# 512차원 모델로 새로 로드
model_512 = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor_512 = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def get_text_embedding(query_text):
    inputs = processor_512(text=[query_text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        # model.get_text_features returns a BaseModelOutputWithPooling object.
        # The actual text embedding is in its 'pooler_output' attribute.
        output = model_512.get_text_features(**inputs)
        embedding = output.pooler_output
    embedding = embedding / embedding.norm(dim=-1, keepdim=True)
    return embedding.cpu().numpy()

def search_category(query_text, index, paths, top_k=5):
    if index.ntotal == 0:
        return []
    top_k = min(top_k, index.ntotal)
    text_vec = get_text_embedding(query_text)
    scores, indices = index.search(text_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({"path": paths[idx], "score": float(score)})
    return results

def get_outfit_combinations(query: str):
    print(f"입력: {query}")
    keywords = extract_fashion_keywords(query)
    text_query = ", ".join(keywords)
    print(f"키워드: {keywords}\n")

    tops = search_category(text_query, index_top, top_paths)
    bottoms = search_category(text_query, index_bottom, bottom_paths)
    dresses = search_category(text_query, index_dress, dress_paths)
    outers = search_category(text_query, index_outer, outer_paths)

    dress_keywords = ["wedding", "guest", "formal", "elegant", "dress"]
    include_dress = any(k in text_query.lower() for k in dress_keywords)

    combinations = []
    for top in tops:
        for bottom in bottoms:
            combo_score = (top["score"] + bottom["score"]) / 2
            combinations.append({
                "top": top["path"],
                "bottom": bottom["path"],
                "score": combo_score
            })

    combinations.sort(key=lambda x: -x["score"])
    top3 = combinations[:3]

    print("👗 추천 코디 Top 3:")
    for i, combo in enumerate(top3):
        print(f"\n  {i+1}번 코디 (점수: {combo['score']:.3f})")
        print(f"  상의: {combo['top']}")
        print(f"  하의: {combo['bottom']}")

    if include_dress and dresses:
        print(f"\n  + 원피스 추천: {dresses[0]['path']} (점수: {dresses[0]['score']:.3f})")

    if outers:
        print(f"\n  + 아우터 추천: {outers[0]['path']} (점수: {outers[0]['score']:.3f})")

    return top3

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [24]:
get_outfit_combinations("결혼식 하객룩 추천해줘")

입력: 결혼식 하객룩 추천해줘
키워드: ['formal', 'elegant', 'luxurious', 'bright color', 'wedding', 'guest', 'dressy', 'sophisticated', 'classy']

👗 추천 코디 Top 3:

  1번 코디 (점수: 0.142)
  상의: /content/drive/MyDrive/패션시멘틱검색/옷/캐주얼위.webp
  하의: /content/drive/MyDrive/패션시멘틱검색/옷/청바지.webp

  + 원피스 추천: /content/drive/MyDrive/패션시멘틱검색/옷/꽃.jpg (점수: 0.232)

  + 아우터 추천: /content/drive/MyDrive/패션시멘틱검색/옷/아디다스 상의.webp (점수: 0.125)


[{'top': '/content/drive/MyDrive/패션시멘틱검색/옷/캐주얼위.webp',
  'bottom': '/content/drive/MyDrive/패션시멘틱검색/옷/청바지.webp',
  'score': 0.14211618900299072}]